# 🎯 ML Classification Project: Customer Churn Prediction

**Goal:** Build and evaluate supervised classification models to predict customer churn.

**Algorithms compared:** Logistic Regression vs. Random Forest  
**Dataset:** Synthetic telco churn dataset (1,000 samples, 15 features)  
**Evaluation:** Accuracy, Precision, Recall, F1-score, ROC-AUC with 5-fold cross-validation

---


## 1. Setup & Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, learning_curve
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, classification_report
)

print("All libraries imported successfully ✓")


## 2. Data Generation & Exploration

In [ ]:
# Generate synthetic churn dataset
np.random.seed(42)
X_raw, y = make_classification(
    n_samples=1000, n_features=15, n_informative=10,
    n_redundant=3, n_classes=2, weights=[0.65, 0.35],
    flip_y=0.05, random_state=42
)

feature_names = [
    "tenure_months", "monthly_charges", "total_charges", "num_products",
    "support_calls", "contract_type", "payment_method", "online_security",
    "tech_support", "streaming_tv", "paperless_billing", "senior_citizen",
    "partner", "dependents", "internet_service"
]

df = pd.DataFrame(X_raw, columns=feature_names)
df["churn"] = y

print(f"Shape: {df.shape}")
print(f"Churn rate: {y.mean()*100:.1f}%")
print(f"\nClass counts:\n{df['churn'].value_counts()}")
df.describe().round(2)


## 3. EDA — Class Distribution, Correlations, Feature Importance

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("Exploratory Data Analysis", fontsize=15, fontweight="bold")

# Class distribution
ax = axes[0]
counts = pd.Series(y).value_counts().sort_index()
ax.bar(["No Churn", "Churn"], counts.values, color=["#4F8EF7", "#F75C5C"], width=0.5)
ax.set_title("Class Distribution"); ax.set_ylabel("Count")

# Correlation heatmap
ax = axes[1]
all_feats = feature_names + ["churn"]
corr = df[all_feats].corr()
top8 = corr["churn"].abs().drop("churn").nlargest(8).index.tolist() + ["churn"]
sns.heatmap(df[top8].corr(), ax=ax, annot=True, fmt=".2f", cmap="coolwarm",
            center=0, linewidths=0.5, annot_kws={"size": 8})
ax.set_title("Top 8 Feature Correlations")
ax.tick_params(axis='x', rotation=45, labelsize=8)

# Placeholder for RF feature importance (filled after model training)
axes[2].set_title("Feature Importance (after model fit)")
axes[2].text(0.5, 0.5, "Run model cells first", ha="center", va="center",
             transform=axes[2].transAxes, fontsize=12, color="gray")

plt.tight_layout(); plt.show()


## 4. Preprocessing & Train/Test Split

In [ ]:
X = df.drop("churn", axis=1).values
y = df["churn"].values

# Stratified 80/20 split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Feature scaling (for Logistic Regression)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

print(f"Train size: {X_train.shape[0]} samples")
print(f"Test size:  {X_test.shape[0]} samples")
print(f"Train churn rate: {y_train.mean()*100:.1f}%")
print(f"Test churn rate:  {y_test.mean()*100:.1f}%")


## 5. Model Definition & 5-Fold Cross-Validation

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Random Forest":       RandomForestClassifier(n_estimators=150, max_depth=8,
                                                   min_samples_leaf=5, random_state=42)
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_results = {}

print("5-Fold Cross-Validation (ROC-AUC):")
print("-" * 45)
for name, model in models.items():
    Xtr = X_train_s if name == "Logistic Regression" else X_train
    scores = cross_val_score(model, Xtr, y_train, cv=cv, scoring="roc_auc")
    cv_results[name] = scores
    print(f"{name:<25}: {scores.mean():.4f} ± {scores.std():.4f}")


## 6. Fit Models & Evaluate on Test Set

In [ ]:
results, cms, fpr_d, tpr_d, auc_d = {}, {}, {}, {}, {}

for name, model in models.items():
    Xtr = X_train_s if name == "Logistic Regression" else X_train
    Xte = X_test_s  if name == "Logistic Regression" else X_test
    
    model.fit(Xtr, y_train)
    y_pred = model.predict(Xte)
    y_prob = model.predict_proba(Xte)[:, 1]
    
    results[name] = {
        "Accuracy":  accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall":    recall_score(y_test, y_pred),
        "F1":        f1_score(y_test, y_pred),
        "ROC-AUC":   roc_auc_score(y_test, y_prob),
    }
    cms[name] = confusion_matrix(y_test, y_pred)
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    fpr_d[name], tpr_d[name], auc_d[name] = fpr, tpr, results[name]["ROC-AUC"]
    
    print(f"\n=== {name} ===")
    print(classification_report(y_test, y_pred, target_names=["No Churn","Churn"]))

pd.DataFrame(results).T.round(4)


## 7. Visualisations — ROC, Confusion Matrices, Metrics Bar Chart

In [ ]:
import matplotlib.gridspec as gridspec

fig = plt.figure(figsize=(18, 12))
gs  = gridspec.GridSpec(2, 3, hspace=0.45, wspace=0.35)
fig.suptitle("Model Evaluation Dashboard", fontsize=16, fontweight="bold")

MODEL_COLORS = {"Logistic Regression": "#4F8EF7", "Random Forest": "#34C98B"}
metric_names = ["Accuracy","Precision","Recall","F1","ROC-AUC"]
x, w = np.arange(len(metric_names)), 0.32

# Metrics bar
ax1 = fig.add_subplot(gs[0, :2])
for i, (name, color) in enumerate(MODEL_COLORS.items()):
    vals = [results[name][m] for m in metric_names]
    bars = ax1.bar(x+(i-0.5)*w, vals, w, label=name, color=color, alpha=0.9)
    for bar, val in zip(bars, vals):
        ax1.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.004, f"{val:.3f}",
                 ha="center", va="bottom", fontsize=8.5, fontweight="bold")
ax1.set_xticks(x); ax1.set_xticklabels(metric_names); ax1.set_ylim(0, 1.15)
ax1.set_ylabel("Score"); ax1.set_title("Test Metrics Comparison"); ax1.legend()

# ROC curves
ax2 = fig.add_subplot(gs[0, 2])
for name, color in MODEL_COLORS.items():
    ax2.plot(fpr_d[name], tpr_d[name], lw=2.5, color=color, label=f"{name} ({auc_d[name]:.3f})")
ax2.plot([0,1],[0,1],"k--",lw=1,label="Random (0.500)")
ax2.set_xlabel("FPR"); ax2.set_ylabel("TPR"); ax2.set_title("ROC Curves"); ax2.legend(fontsize=8)

# Confusion matrices
for col, (name, color) in enumerate(MODEL_COLORS.items()):
    ax = fig.add_subplot(gs[1, col])
    cm = cms[name]; cm_pct = cm/cm.sum(axis=1,keepdims=True)
    annots = [[f"{cm[i,j]}\n({cm_pct[i,j]*100:.1f}%)" for j in range(2)] for i in range(2)]
    sns.heatmap(cm, ax=ax, annot=annots, fmt="", cmap="Blues",
                xticklabels=["No Churn","Churn"], yticklabels=["No Churn","Churn"],
                linewidths=1, linecolor="white", cbar=False, annot_kws={"size":10})
    ax.set_title(f"Confusion Matrix — {name}"); ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")

# CV Boxplot
ax5 = fig.add_subplot(gs[1, 2])
bp = ax5.boxplot([cv_results[n] for n in MODEL_COLORS], patch_artist=True, widths=0.4)
for patch, color in zip(bp["boxes"], MODEL_COLORS.values()):
    patch.set_facecolor(color); patch.set_alpha(0.8)
ax5.set_xticks([1,2]); ax5.set_xticklabels(list(MODEL_COLORS)); ax5.set_ylabel("ROC-AUC")
ax5.set_title("5-Fold CV Distribution"); ax5.set_ylim(0.5,1.0)

plt.show()


## 8. Learning Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Learning Curves (ROC-AUC vs Training Size)", fontsize=14, fontweight="bold")

for ax, (name, model), color in zip(axes, models.items(),
                                    ["#4F8EF7","#34C98B"]):
    Xall = X_train_s if name == "Logistic Regression" else X_train
    ts, tr_m, va_m = learning_curve(model, Xall, y_train,
                                     train_sizes=np.linspace(0.1,1.0,10),
                                     cv=cv, scoring="roc_auc", n_jobs=-1)
    tr_mean, tr_std = tr_m.mean(1), tr_m.std(1)
    va_mean, va_std = va_m.mean(1), va_m.std(1)
    ax.plot(ts, tr_mean, color=color,    lw=2.5, label="Train")
    ax.plot(ts, va_mean, color="#F5A623", lw=2.5, label="Validation", linestyle="--")
    ax.fill_between(ts, tr_mean-tr_std, tr_mean+tr_std, alpha=0.15, color=color)
    ax.fill_between(ts, va_mean-va_std, va_mean+va_std, alpha=0.15, color="#F5A623")
    ax.set_title(name); ax.set_xlabel("Training Samples")
    ax.set_ylabel("ROC-AUC"); ax.legend(); ax.set_ylim(0.5, 1.05)
plt.tight_layout(); plt.show()


## 9. Summary Table

In [ ]:
summary = pd.DataFrame(results).T.round(4)
summary.index.name = "Model"
summary.style.background_gradient(cmap="RdYlGn", axis=0).format("{:.4f}")
